In [1]:
import math
import torch
import torch.nn.functional as F
import torch.nn as nn

#mask 행렬 생성
def generate_mask(seq_len):
    return torch.triu(
        torch.ones(seq_len, seq_len),
        diagonal=1
    )

#MHA
def scaled_dot_product_attention(Q,K,V, mask = None):
    #Q,K,V: (batch, heads, seq_len, d_k) 크기의 행렬
    scores = torch.matmul(Q,K.transpose(-2,-1)) / math.sqrt(Q.size(-1))

    if mask is not None:
        #Decoder를 위한 mask 수정
        scores = scores.masked_fill(mask, float("-inf"))

    attn = F.softmax(scores,dim=-1)
    output = torch.matmul(attn,V)

    return output, attn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0 #head마다 같은 차원을 갖도록 나누어 떨어지는지 확인

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B, L_q, _ = q.shape
        _, L_k, _ = k.shape
        Q = self.W_q(q).view(B, L_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)

                # Mask 차원 맞추기
        if mask is not None:

            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)

            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)

            # bool 타입으로 변환
            mask = mask.bool()

        out, attn = scaled_dot_product_attention(Q, K, V, mask)

        out = out.transpose(1, 2).contiguous().view(B, L_q, self.d_model)
        out = self.W_o(out)

        return out, attn

#AddNorm
class AddNorm(nn.Module):
    def __init__(self,d_model,dropout = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)


    def forward(self,x,sublayer_out):
        return self.norm(x+self.dropout(sublayer_out))


#FeedForward
class FeedForward(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model,d_ff) #512 -> 2048
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(d_ff,d_model) #2048 -> 512

    def forward(self,x):
        return self.fc2(self.act(self.fc1(x)))


#PE
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len = 5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
    
        position = torch.arange(max_len).unsqueeze(1)
    
        div_term = torch.exp(
            torch.arange(0,d_model,2)*(-math.log(10000.0)/d_model)
        )
    
        pe[:,0::2] = torch.sin(position * div_term)
    
        pe[:, 1::2] = torch.cos(position * div_term)
    
        pe = pe.unsqueeze(0)
    
        self.register_buffer("pe", pe)

    
    def forward(self, x):
        seq_len = x.size(1)
    
        x = x + self.pe[:,:seq_len]
    
        return x

In [2]:
#Encoder Part
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)

        self.addnorm1 = AddNorm(d_model, dropout)
        self.addnorm2 = AddNorm(d_model, dropout)

    def forward(self, x, mask = None):
        out, attn = self.mha(x,x,x, mask)
        x = self.addnorm1(x, out)

        ffn_out = self.ffn(x)
        x = self.addnorm2(x,ffn_out)

        return x, attn



class Encoder(nn.Module):

    def __init__(self,d_model, num_heads,d_ff,num_layers, dropout = 0.1):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):

        attn = None

        for layer in self.layers:
            x, attn = layer(x, mask)

        return x, attn


In [3]:
#Decoder Part
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        
        self.ffn = FeedForward(d_model, d_ff)

        self.addnorm1 = AddNorm(d_model, dropout)
        self.addnorm2 = AddNorm(d_model, dropout)
        self.addnorm3 = AddNorm(d_model, dropout)
        
    def forward(self, x, encoder_output, mask = None):
        seq_len = x.size(1)
        mask = generate_mask(seq_len).to(x.device)
        
        self_out, self_attn = self.self_attn(x,x,x, mask)
        x = self.addnorm1(x, self_out)

        cross_output, cross_attn = self.cross_attn(x,encoder_output,encoder_output)
        x = self.addnorm2(x,cross_output)

        ffn_out = self.ffn(x)
        x = self.addnorm3(x, ffn_out)

        return x, self_attn, cross_attn

class Decoder(nn.Module):

    def __init__(self,d_model, num_heads,d_ff,num_layers, dropout = 0.1):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

    def forward(self, x, encoder_output, mask=None):

        self_attn = None
        cross_attn = None

        for layer in self.layers:
            x, self_attn, cross_attn = layer(x, encoder_output)

        return x, self_attn, cross_attn

In [4]:
#Transformer
class Transformer(nn.Module):
    def __init__( self,src_vocab_size, tgt_vocab_size, d_model, num_heads,
        d_ff,num_layers,max_len=5000, dropout=0.1 ):

        super().__init__()

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        self.positional_encoding = PositionalEncoding(d_model, max_len)


        #Encoder
        self.encoder = Encoder(d_model, num_heads,d_ff,num_layers, dropout)

        #Decoder
        self.decoder = Decoder(d_model, num_heads,d_ff,num_layers, dropout = 0.1)

        self.fc = nn.Linear(d_model, tgt_vocab_size)
    def forward(self, src, tgt):

        src = self.src_embedding(src)
        src = self.positional_encoding(src)

        encoder_output, encoder_attn = self.encoder(src)

        tgt = self.tgt_embedding(tgt)
        tgt = self.positional_encoding(tgt)

        decoder_output, self_attn, cross_attn = self.decoder(tgt,encoder_output)

        output = self.fc(decoder_output)

        return output, encoder_attn, self_attn, cross_attn

In [5]:
#Train
PAD = 0
BOS = 1
EOS = 2

src_data = torch.tensor([
    [3, 4, 5],
    [3, 5, 4],
    [4, 3, 5],
    [4, 5, 3],
    [5, 3, 4],
    [5, 4, 3]
])

tgt_data = torch.tensor([
    [6, 7, 8],
    [6, 8, 7],
    [7, 6, 8],
    [7, 8, 6],
    [8, 6, 7],
    [8, 7, 6]
])
#3 → 6 4 → 7 5 → 8

tgt_input = torch.tensor([
    [BOS, 6, 7, 8],
    [BOS, 6, 8, 7],
    [BOS, 7, 6, 8],
    [BOS, 7, 8, 6],
    [BOS, 8, 6, 7],
    [BOS, 8, 7, 6]
])

tgt_output = torch.tensor([
    [6, 7, 8, EOS],
    [6, 8, 7, EOS],
    [7, 6, 8, EOS],
    [7, 8, 6, EOS],
    [8, 6, 7, EOS],
    [8, 7, 6, EOS]
])

src_vocab_size = 10
tgt_vocab_size = 10

d_model = 32
num_heads = 4
d_ff = 64
num_layers = 2

model = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    dropout=0.1
)

import torch.optim as optim
loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(),lr=1e-3)

num_epochs = 1000

model.train()

for epoch in range(num_epochs):

    # Forward
    output, _, _, _ = model(
        src_data,
        tgt_input
    )

    # output:
    # (batch_size, seq_len, vocab_size)

    # tgt_output:
    # (batch_size, seq_len)

    loss = loss_fn(
        output.reshape(-1, tgt_vocab_size),
        tgt_output.reshape(-1)
    )

    # Gradient 초기화
    optimizer.zero_grad()

    loss.backward()

    # Parameter 업데이트
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch}, Loss: {loss.item():.4f}"
        )


Epoch 0, Loss: 2.6119
Epoch 100, Loss: 0.0658
Epoch 200, Loss: 0.0139
Epoch 300, Loss: 0.0073
Epoch 400, Loss: 0.0045
Epoch 500, Loss: 0.0032
Epoch 600, Loss: 0.0023
Epoch 700, Loss: 0.0018
Epoch 800, Loss: 0.0015
Epoch 900, Loss: 0.0012


In [6]:
model.eval()

with torch.no_grad():
    output, _, _, _ = model(src_data, tgt_input)

pred = output.argmax(dim=-1)

print(pred)
print(tgt_output)

tensor([[6, 7, 8, 2],
        [6, 8, 7, 2],
        [7, 6, 8, 2],
        [7, 8, 6, 2],
        [8, 6, 7, 2],
        [8, 7, 6, 2]])
tensor([[6, 7, 8, 2],
        [6, 8, 7, 2],
        [7, 6, 8, 2],
        [7, 8, 6, 2],
        [8, 6, 7, 2],
        [8, 7, 6, 2]])


In [7]:
def greedy_decode(model, src, max_len, bos_token, eos_token):

    model.eval()

    with torch.no_grad():
        # 1. Encoder
        src_emb = model.src_embedding(src)
        src_emb = model.positional_encoding(src_emb)

        encoder_output, _ = model.encoder(src_emb)

        # 2. BOS로 시작
        tgt = torch.tensor(
            [[bos_token]],
            device=src.device
        )

        # 3. 한 토큰씩 생성
        for _ in range(max_len - 1):

            tgt_emb = model.tgt_embedding(tgt)
            tgt_emb = model.positional_encoding(tgt_emb)

            decoder_output, _, _ = model.decoder(
                tgt_emb,
                encoder_output
            )

            # 마지막 위치의 출력만 사용
            logits = model.fc(decoder_output[:, -1, :])

            # 가장 높은 점수의 토큰 선택
            next_token = logits.argmax(dim=-1).item()

            # 생성된 토큰 추가
            tgt = torch.cat([
                tgt,
                torch.tensor(
                    [[next_token]],
                    device=src.device
                )
            ], dim=1)

            # EOS가 나오면 종료
            if next_token == eos_token:
                break

        return tgt

In [8]:
src = src_data[0].unsqueeze(0)

result = greedy_decode(
    model,
    src,
    max_len=5,
    bos_token=BOS,
    eos_token=EOS
)

print(result)

tensor([[1, 6, 7, 8, 2]])
